In [1]:
# SHAP Explainability Analysis
# Multi-Target Price Forecasting using LightGBM

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lightgbm as lgb
from sklearn.multioutput import MultiOutputRegressor

import shap

warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (8,6)
plt.rcParams["font.size"] = 11

print("SHAP EXPLAINABILITY ANALYSIS")

/Users/bruker/Desktop/Multi-Target Price Forecasting /.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SHAP EXPLAINABILITY ANALYSIS


In [2]:
data_path = os.path.join("data","all_months_features.csv")

df_raw = pd.read_csv(data_path)

print(df_raw.shape)
df_raw.head()

(798, 77)


,Product_Name,Category,bs_year,bs_month,month_idx,month_name,Volume,Min_Price,Max_Price,Avg_Price,...,price_spread,price_spread_ratio,Volume_lag1,TOTAL_sources_lag1,volume_change_ratio,sin_month,cos_month,total_imports,import_ratio,ktm_supply_share
0,Akabary_Chilly,vegetable,2082,9,6,poush,880,400.0,600.0,500.00,...,200.0,0.400000,NaN,NaN,NaN,-5.877853e-01,-0.809017,0,0.000000,0.0
1,Akabary_Chilly,vegetable,2082,10,7,magh,540,300.0,900.0,628.57,...,600.0,0.954548,880.0,880.0,-0.386364,-9.510565e-01,-0.309017,0,0.000000,0.0
2,Akabary_Chilly,vegetable,2082,11,8,falgun,0,300.0,800.0,555.56,...,500.0,0.899993,540.0,540.0,-1.000000,-9.510565e-01,0.309017,0,0.000000,0.0
3,Akabary_Chilly,vegetable,2082,12,9,chaitra,5920,300.0,500.0,439.77,...,200.0,0.454783,0.0,0.0,NaN,-5.877853e-01,0.809017,5260,0.888514,0.0
4,Akabary_Chilly,vegetable,2083,1,10,baishakh,13770,400.0,500.0,450.00,...,100.0,0.222222,5920.0,5920.0,1.326014,-2.449294e-16,1.000000,3970,0.288308,0.0


In [3]:
# Create Next Month Targets
df = df_raw.copy()

df["month_idx"] = (
    df.groupby("Product_Name")
      .cumcount()+1
)

df = df.sort_values(
    ["Product_Name","month_idx"]
).reset_index(drop=True)

targets_base = [
    "Min_Price",
    "Avg_Price",
    "Max_Price"
]

targets_next = [
    "Min_Price_next",
    "Avg_Price_next",
    "Max_Price_next"
]

for base,next_col in zip(targets_base,targets_next):

    df[next_col] = (
        df.groupby("Product_Name")[base]
        .shift(-1)
    )

In [4]:
# Clean dataset
df_clean = df.dropna(
    subset=["Avg_Price_next"]
).copy()

df_clean["Min_Price_next"] = (
    df_clean["Min_Price_next"]
    .fillna(df_clean["Avg_Price_next"])
)

df_clean["Max_Price_next"] = (
    df_clean["Max_Price_next"]
    .fillna(df_clean["Avg_Price_next"])
)

In [5]:
# Feature Selection
non_feature_cols = [

    "Product_Name",
    "Category",
    "Unit",
    "unit_canonical",
    "month_name",
    "bs_year",
    "bs_month",

    "Min_Price",
    "Avg_Price",
    "Max_Price",

    "Min_Price_next",
    "Avg_Price_next",
    "Max_Price_next",

    "Total_Amount"

]

feature_cols = [
    c for c in df_clean.columns
    if c not in non_feature_cols
]

X = (
    df_clean[feature_cols]
    .apply(pd.to_numeric,errors="coerce")
    .fillna(0)
)

y = df_clean[targets_next].fillna(0)

print("Features:",len(feature_cols))

Features: 66


In [6]:
# Time-based Split: Train (<=7), Valid (==8), Test (==9)
train_mask = df_clean["month_idx"]<=7
valid_mask = df_clean["month_idx"]==8
test_mask  = df_clean["month_idx"]==9

X_train = X[train_mask]
X_valid = X[valid_mask]
X_test  = X[test_mask]

y_train = y[train_mask]
y_valid = y[valid_mask]
y_test  = y[test_mask]

In [7]:
lgb_model = lgb.LGBMRegressor(

    objective="regression_l1",

    learning_rate=0.05,

    num_leaves=31,

    n_estimators=600,

    random_state=42,

    n_jobs=-1,

    verbose=-1

)

model = MultiOutputRegressor(lgb_model)

model.fit(X_train,y_train)

print("Training Finished.")

Training Finished.


In [8]:
save_dir = "shap_plots"

os.makedirs(save_dir,exist_ok=True)

In [9]:
# Shap Analysis
target_names = [
    "min",
    "avg",
    "max"
]

shap_values_all = {}
explainers = {}

for name,estimator in zip(target_names,model.estimators_):

    print("Processing:",name)

    explainer = shap.TreeExplainer(estimator)

    shap_values = explainer(X_test)

    explainers[name] = explainer

    shap_values_all[name] = shap_values

Processing: min
Processing: avg
Processing: max


In [10]:
# Figure 1 Global Feature Importance
for name in target_names:

    plt.figure(figsize=(8,6))

    shap.plots.bar(
        shap_values_all[name],
        show=False
    )

    plt.title(f"Global SHAP Importance ({name.upper()})")

    plt.tight_layout()

    plt.savefig(
        f"{save_dir}/Figure1_Global_{name}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

In [11]:
# Figure 2 Beeswarm
for name in target_names:

    plt.figure(figsize=(8,6))

    shap.plots.beeswarm(
        shap_values_all[name],
        show=False
    )

    plt.title(f"SHAP Beeswarm ({name.upper()})")

    plt.tight_layout()

    plt.savefig(
        f"{save_dir}/Figure2_Beeswarm_{name}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

In [12]:
# Figure 3 Dependence Plot
feature_name = "Avg_Price_lag1"

if feature_name not in feature_cols:

    feature_name = feature_cols[0]

print("Dependence Feature:",feature_name)

Dependence Feature: Avg_Price_lag1


In [13]:
for name in target_names:

    plt.figure(figsize=(8,6))

    shap.plots.scatter(

        shap_values_all[name][:,feature_name],

        color=shap_values_all[name],

        show=False

    )

    plt.title(
        f"Dependence Plot ({feature_name}) - {name.upper()}"
    )

    plt.tight_layout()

    plt.savefig(

        f"{save_dir}/Figure3_Dependence_{name}.png",

        dpi=300,

        bbox_inches="tight"

    )

    plt.close()

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

In [14]:
# Figure 4 Waterfall Plot
sample_index = 0

for name in target_names:

    plt.figure(figsize=(9,6))

    shap.plots.waterfall(

        shap_values_all[name][sample_index],

        show=False

    )

    plt.title(
        f"Waterfall Plot ({name.upper()})"
    )

    plt.tight_layout()

    plt.savefig(

        f"{save_dir}/Figure4_Waterfall_{name}.png",

        dpi=300,

        bbox_inches="tight"

    )

    plt.close()

In [15]:
# Global Feature Importance CSV
for name in target_names:

    importance = pd.DataFrame({

        "Feature":feature_cols,

        "MeanAbsSHAP":
        np.abs(shap_values_all[name].values).mean(axis=0)

    })

    importance = importance.sort_values(

        "MeanAbsSHAP",

        ascending=False

    )

    importance.to_csv(

        f"{save_dir}/GlobalImportance_{name}.csv",

        index=False

    )

In [17]:
for name in target_names:

    print("\n")

    print(name.upper())

    importance = pd.read_csv(

        f"{save_dir}/GlobalImportance_{name}.csv"

    )

    display(importance.head(10))



MIN


,Feature,MeanAbsSHAP
0,Avg_Price_lag1,27.828798
1,Local,10.216470
2,price_spread,10.105926
3,price_spread_ratio,9.487885
4,Min_Price_roll3_mean,7.421453
5,Narayangadh,6.956697
6,Volume,5.743488
7,Avg_Price_roll3_mean,5.679480
8,Min_Price_lag1,4.150731
9,import_share,3.853760




AVG


,Feature,MeanAbsSHAP
0,Avg_Price_lag1,27.608728
1,price_spread,16.101904
2,Local,10.251315
3,Min_Price_lag1,8.303529
4,Max_Price_lag1,7.615450
5,price_spread_ratio,7.613535
6,Narayangadh,6.532351
7,Volume,4.949072
8,Avg_Price_roll3_mean,4.929158
9,China,3.366881




MAX


,Feature,MeanAbsSHAP
0,Avg_Price_lag1,32.981318
1,price_spread,27.861326
2,price_spread_ratio,10.067506
3,Min_Price_lag1,8.832788
4,Max_Price_lag1,7.902783
5,Local,6.492588
6,Avg_Price_roll3_mean,5.062752
7,Max_Price_roll3_mean,4.234683
8,Avg_Price_roll3_std,4.160764
9,Volume,3.532402


In [18]:
print("\nSHAP analysis completed successfully.")

print("\nFiles saved inside:")

print(save_dir)


SHAP analysis completed successfully.

Files saved inside:
shap_plots
